In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **download_video**

In [ ]:
# ==========================================
# Stage 1 - Video Downloader (Fixed)
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

import os, asyncio
from telethon import TelegramClient
import gspread

API_ID    = int(os.environ.get("API_ID", 0))  # set in Colab Secrets
API_HASH  = os.environ.get("API_HASH", "")   # set in Colab Secrets
BOT_TOKEN = os.environ.get("BOT_TOKEN", "")  # set in Colab Secrets

BASE_DIR   = "/content/drive/MyDrive/subtitle_bot"
VIDEOS_DIR = os.path.join(BASE_DIR, "videos")
CRED_FILE  = os.path.join(BASE_DIR, "credentials.json")

gc        = gspread.service_account(filename=CRED_FILE)
sh        = gc.open("subtitle_jobs")
worksheet = sh.worksheet("jobs")

COL_STATUS     = 5   
COL_VIDEO_PATH = 6   


async def main():
    all_jobs = worksheet.get_all_records()

    new_jobs = [j for j in all_jobs if str(j.get("status", "")).strip() == "new"]
    print(f"Total jobs: {len(all_jobs)} | New jobs to download: {len(new_jobs)}")

    if not new_jobs:
        print("No new jobs. Exiting.")
        return

    session_path = '/content/drive/MyDrive/subtitle_bot/logs/bot_session'
    client = TelegramClient(session_path, API_ID, API_HASH)

    try:
        await client.start(bot_token=BOT_TOKEN)
        print("Bot connected successfully!")

        for job in new_jobs:
            job_id    = str(job['job_id'])
            chat_id   = int(job['chat_id'])
            message_id = int(job['message_id'])

            
            row_idx = all_jobs.index(job) + 2   



            try:
                message = await client.get_messages(chat_id, ids=message_id)

                if not message or not message.media:
                    print(f"[ERROR] No media for job {job_id}")
                    worksheet.update_cell(row_idx, COL_STATUS, "error_no_media")
                    continue

                
                ext = ".mp4"
                if message.file:
                    mime = getattr(message.file, 'mime_type', '') or ''
                    name = getattr(message.file, 'name', '') or ''
                    if "mov" in mime:
                        ext = ".mov"
                    elif "mkv" in mime:
                        ext = ".mkv"
                    elif name:
                        ext = os.path.splitext(name)[1] or ".mp4"

                video_filename = f"video_{job_id}{ext}"
                video_path     = os.path.join(VIDEOS_DIR, video_filename)

               
                partial = video_path + ".partial"
                if os.path.exists(partial):
                    os.remove(partial)
                    print(f"[CLEAN] Removed partial file for job {job_id}")

                print(f"[DOWNLOAD] job {job_id} → {video_filename}")
                worksheet.update_cell(row_idx, COL_STATUS, "downloading")

                await message.download_media(file=video_path)
                print(f"[DONE] {video_path}")

                
                worksheet.update_cell(row_idx, COL_STATUS,     "downloaded")
                worksheet.update_cell(row_idx, COL_VIDEO_PATH, video_path)

            except Exception as e:
                print(f"[ERROR] job {job_id}: {e}")
                import traceback; traceback.print_exc()
                worksheet.update_cell(row_idx, COL_STATUS, "error_download")

    finally:
        await client.disconnect()
        print("Client disconnected.")


asyncio.run(main())
print("Stage 1 finished.")

# **Audio Extractor**

In [ ]:
# ==========================================
# Stage 2 - Audio Extractor (Fixed)
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
import gspread

BASE_DIR    = "/content/drive/MyDrive/subtitle_bot"
VIDEOS_DIR  = os.path.join(BASE_DIR, "videos")
AUDIOS_DIR  = os.path.join(BASE_DIR, "audios")
CRED_FILE   = os.path.join(BASE_DIR, "credentials.json")

gc        = gspread.service_account(filename=CRED_FILE)
sh        = gc.open("subtitle_jobs")
worksheet = sh.worksheet("jobs")


COL_STATUS = 5   

all_jobs = worksheet.get_all_records()

ready_jobs = [j for j in all_jobs if str(j.get("status", "")).strip() == "downloaded"]
print(f"Total jobs: {len(all_jobs)} | Ready for audio extraction: {len(ready_jobs)}")

for job in ready_jobs:
    job_id  = str(job['job_id'])
    row_idx = all_jobs.index(job) + 2

    audio_path = os.path.join(AUDIOS_DIR, f"audio_{job_id}.wav")

    if os.path.exists(audio_path):
        print(f"[SKIP] Audio for job {job_id} already exists.")
        worksheet.update_cell(row_idx, COL_STATUS, "audio_extracted")
        continue

    
    video_files = [f for f in os.listdir(VIDEOS_DIR) if f.startswith(f"video_{job_id}.")]
    if not video_files:
        print(f"[WARN] No video file found for job {job_id}")
        worksheet.update_cell(row_idx, COL_STATUS, "error_no_video")
        continue

    video_path = os.path.join(VIDEOS_DIR, video_files[0])
    print(f"[EXTRACT] job {job_id}")

    cmd = [
        "ffmpeg", "-y", "-i", video_path,
        "-vn", "-acodec", "pcm_s16le",
        "-ar", "16000", "-ac", "1",
        audio_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"[ERROR] FFmpeg failed for job {job_id}\n{result.stderr}")
        worksheet.update_cell(row_idx, COL_STATUS, "error_audio")
        continue

    print(f"[DONE] {audio_path}")
    worksheet.update_cell(row_idx, COL_STATUS, "audio_extracted")

print("Stage 2 finished.")

# **text Extractor & Transcriber**

In [ ]:
# ==========================================
# Stage 3 - Transcriber & Translator (Fixed)
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

import os
import gspread
from faster_whisper import WhisperModel
from deep_translator import GoogleTranslator

BASE_DIR       = "/content/drive/MyDrive/subtitle_bot"
AUDIOS_DIR     = os.path.join(BASE_DIR, "audios")
SUBTITLES_DIR  = os.path.join(BASE_DIR, "subtitles")
MODEL_DIR      = os.path.join(BASE_DIR, "models", "large-v3")
CRED_FILE      = os.path.join(BASE_DIR, "credentials.json")

gc        = gspread.service_account(filename=CRED_FILE)
sh        = gc.open("subtitle_jobs")
worksheet = sh.worksheet("jobs")

COL_STATUS   = 5   
COL_SRT_PATH = 7   

all_jobs = worksheet.get_all_records()

ready_jobs = [j for j in all_jobs if str(j.get("status", "")).strip() == "audio_extracted"]
print(f"Total jobs: {len(all_jobs)} | Ready for transcription: {len(ready_jobs)}")

if not ready_jobs:
    print("No jobs ready for transcription.")
else:
    model      = WhisperModel(MODEL_DIR, device="cuda", compute_type="float16")
    translator = GoogleTranslator(source='auto', target='fa')

    def format_time(t):
        hours   = int(t // 3600)
        minutes = int((t % 3600) // 60)
        secs    = int(t % 60)
        millis  = int((t - int(t)) * 1000)
        return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"

    for job in ready_jobs:
        job_id  = str(job['job_id'])
        row_idx = all_jobs.index(job) + 2

        srt_path   = os.path.join(SUBTITLES_DIR, f"sub_{job_id}.srt")
        audio_path = os.path.join(AUDIOS_DIR,    f"audio_{job_id}.wav")

        if os.path.exists(srt_path):
            print(f"[SKIP] SRT for job {job_id} already exists.")
            worksheet.update_cell(row_idx, COL_STATUS,   "transcribed")
            worksheet.update_cell(row_idx, COL_SRT_PATH, srt_path)
            continue

        if not os.path.exists(audio_path):
            print(f"[ERROR] Audio not found for job {job_id}")
            worksheet.update_cell(row_idx, COL_STATUS, "error_no_audio")
            continue

        print(f"[TRANSCRIBE] job {job_id}")
        worksheet.update_cell(row_idx, COL_STATUS, "transcribing")

        try:
            segments, info = model.transcribe(
                audio_path, beam_size=5, language=None, task="transcribe"
            )
            print(f"  Detected language: {info.language} ({info.language_probability:.2f})")

            srt_lines = []
            idx = 1
            for segment in segments:
                text = segment.text.strip()
                if not text:
                    continue
                try:
                    translated = translator.translate(text)
                except Exception as e:
                    print(f"  [WARN] Translation failed: {e} — using original")
                    translated = text

                srt_lines.append(str(idx))
                srt_lines.append(f"{format_time(segment.start)} --> {format_time(segment.end)}")
                srt_lines.append(translated)
                srt_lines.append("")
                idx += 1

            with open(srt_path, 'w', encoding='utf-8') as f:
                f.write("\n".join(srt_lines))

            print(f"[DONE] {srt_path}")
            worksheet.update_cell(row_idx, COL_STATUS,   "transcribed")
            worksheet.update_cell(row_idx, COL_SRT_PATH, srt_path)

        except Exception as e:
            print(f"[ERROR] Transcription failed for job {job_id}: {e}")
            import traceback; traceback.print_exc()
            worksheet.update_cell(row_idx, COL_STATUS, "error_transcribe")

print("Stage 3 finished.")

# **Subtitle Renderer**

In [ ]:
# ==========================================
# Stage 4 - Subtitle Renderer (Fixed)
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

import os, re, subprocess
import gspread

BASE_DIR       = "/content/drive/MyDrive/subtitle_bot"
VIDEOS_DIR     = os.path.join(BASE_DIR, "videos")
SUBTITLES_DIR  = os.path.join(BASE_DIR, "subtitles")
OUTPUTS_DIR    = os.path.join(BASE_DIR, "outputs")
FONTS_DIR      = os.path.join(BASE_DIR, "fonts")
CRED_FILE      = os.path.join(BASE_DIR, "credentials.json")

FONT_NAME = "Vazir"

gc        = gspread.service_account(filename=CRED_FILE)
sh        = gc.open("subtitle_jobs")
worksheet = sh.worksheet("jobs")


COL_STATUS      = 5   
COL_OUTPUT_PATH = 8   

all_jobs = worksheet.get_all_records()

ready_jobs = [j for j in all_jobs if str(j.get("status", "")).strip() == "transcribed"]
print(f"Total jobs: {len(all_jobs)} | Ready for rendering: {len(ready_jobs)}")


def srt_time_to_ass(srt_time):
    h, m, s, ms = re.match(
        r"(\d{2}):(\d{2}):(\d{2}),(\d{3})", srt_time
    ).groups()
    return f"{int(h)}:{m}:{s}.{int(ms)//10:02d}"


def srt_to_ass(srt_path, ass_path):
    with open(srt_path, "r", encoding="utf-8") as f:
        content = f.read()

    content = content.replace("\r\n", "\n").strip()
    if not content:
        print(f"[ERROR] Empty subtitle: {srt_path}")
        return False

    pattern = re.compile(
        r"(\d+)\n"
        r"(\d{2}:\d{2}:\d{2},\d{3}) --> "
        r"(\d{2}:\d{2}:\d{2},\d{3})\n"
        r"((?:.*\n?)+?)"
        r"(?=\n\d+\n|\Z)"
    )
    matches = pattern.findall(content)
    if not matches:
        print(f"[ERROR] No subtitle blocks: {srt_path}")
        return False

    ass_header = f"""[Script Info]
ScriptType: v4.00+
WrapStyle: 2
ScaledBorderAndShadow: yes
YCbCr Matrix: TV.709

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,{FONT_NAME},24,&H00FFFFFF,&H00FFFFFF,&H00000000,&H80000000,0,0,0,0,100,100,0,0,1,2.5,2.5,2,50,50,35,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
    events = []
    for _, start, end, text in matches:
        text = text.strip().replace("\n", "\\N")
        events.append(
            f"Dialogue: 0,{srt_time_to_ass(start)},{srt_time_to_ass(end)},Default,,0,0,0,,{text}"
        )

    with open(ass_path, "w", encoding="utf-8") as f:
        f.write(ass_header + "\n".join(events))
    return True


for job in ready_jobs:
    job_id  = str(job['job_id'])
    row_idx = all_jobs.index(job) + 2

    srt_path    = os.path.join(SUBTITLES_DIR, f"sub_{job_id}.srt")
    ass_path    = os.path.join(SUBTITLES_DIR, f"sub_{job_id}.ass")
    output_path = os.path.join(OUTPUTS_DIR,   f"output_{job_id}.mp4")

    if os.path.exists(output_path):
        print(f"[SKIP] output_{job_id}.mp4 already exists")
        worksheet.update_cell(row_idx, COL_STATUS,      "rendered")
        worksheet.update_cell(row_idx, COL_OUTPUT_PATH, output_path)
        continue

    if not os.path.exists(srt_path):
        print(f"[WARN] Missing subtitle: {srt_path}")
        worksheet.update_cell(row_idx, COL_STATUS, "error_no_srt")
        continue

    if os.path.getsize(srt_path) < 10:
        print(f"[WARN] Subtitle too small: {srt_path}")
        worksheet.update_cell(row_idx, COL_STATUS, "error_empty_srt")
        continue

    if not srt_to_ass(srt_path, ass_path):
        worksheet.update_cell(row_idx, COL_STATUS, "error_srt_parse")
        continue

   
    video_files = [f for f in os.listdir(VIDEOS_DIR) if f.startswith(f"video_{job_id}.")]
    if not video_files:
        print(f"[ERROR] No video file for job {job_id}")
        worksheet.update_cell(row_idx, COL_STATUS, "error_no_video")
        continue

    video_path = os.path.join(VIDEOS_DIR, video_files[0])
    print(f"[RENDER] job {job_id}")
    worksheet.update_cell(row_idx, COL_STATUS, "rendering")

    cmd = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-vf", f"ass={ass_path}:fontsdir={FONTS_DIR}",
        "-c:v", "libx264", "-crf", "18", "-preset", "medium",
        "-c:a", "copy",
        output_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"[ERROR] FFmpeg failed for job {job_id}\n{result.stderr}")
        worksheet.update_cell(row_idx, COL_STATUS, "error_render")
        continue

    print(f"[DONE] {output_path}")
    worksheet.update_cell(row_idx, COL_STATUS,      "rendered")
    worksheet.update_cell(row_idx, COL_OUTPUT_PATH, output_path)

print("Stage 4 finished.")

# **Upload video to Telegram**

In [ ]:
# ==========================================
# Stage 5 - Telegram Sender (Fixed)
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

import os, asyncio
from telethon import TelegramClient
import gspread

API_ID    = int(os.environ.get("API_ID", 0))  # set in Colab Secrets
API_HASH  = os.environ.get("API_HASH", "")   # set in Colab Secrets
BOT_TOKEN = os.environ.get("BOT_TOKEN", "")  # set in Colab Secrets

BASE_DIR     = "/content/drive/MyDrive/subtitle_bot"
OUTPUTS_DIR  = os.path.join(BASE_DIR, "outputs")
CRED_FILE    = os.path.join(BASE_DIR, "credentials.json")

gc        = gspread.service_account(filename=CRED_FILE)
sh        = gc.open("subtitle_jobs")
worksheet = sh.worksheet("jobs")


COL_STATUS = 5


async def main():
    all_jobs = worksheet.get_all_records()

    
    ready_jobs = [j for j in all_jobs if str(j.get("status", "")).strip() == "rendered"]
    print(f"Total jobs: {len(all_jobs)} | Ready to send: {len(ready_jobs)}")

    if not ready_jobs:
        print("No jobs ready to send.")
        return

    session_path = '/content/drive/MyDrive/subtitle_bot/logs/bot_session'
    client = TelegramClient(session_path, API_ID, API_HASH)

    try:
        await client.start(bot_token=BOT_TOKEN)
        print("Bot connected successfully!")

        for job in ready_jobs:
            job_id  = str(job['job_id'])
            chat_id = int(job['chat_id'])
            row_idx = all_jobs.index(job) + 2

           
            output_files = [
                f for f in os.listdir(OUTPUTS_DIR)
                if f.startswith(f"output_{job_id}.")
            ]
            if not output_files:
                print(f"[WARN] Output file not found for job {job_id}")
                worksheet.update_cell(row_idx, COL_STATUS, "error_no_output")
                continue

            output_video = os.path.join(OUTPUTS_DIR, output_files[0])

            try:
                print(f"[SEND] job {job_id} → chat {chat_id}")
                worksheet.update_cell(row_idx, COL_STATUS, "sending")

                await client.send_file(
                    chat_id,
                    output_video,
                    caption="🎬 ویدیوی شما با زیرنویس فارسی آماده است."
                )
                print(f"[DONE] Sent job {job_id}")

                
                worksheet.update_cell(row_idx, COL_STATUS, "completed")

            except Exception as e:
                print(f"[ERROR] Failed to send job {job_id}: {e}")
                import traceback; traceback.print_exc()
                worksheet.update_cell(row_idx, COL_STATUS, "error_send")

    finally:
        await client.disconnect()
        print("Client disconnected.")


asyncio.run(main())
print("Stage 5 finished.")

# Cleanup

In [ ]:
# ==========================================
# Stage 6 - Cleanup
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

BASE_DIR = "/content/drive/MyDrive/subtitle_bot"
VIDEOS_DIR = os.path.join(BASE_DIR, "videos")
AUDIOS_DIR = os.path.join(BASE_DIR, "audios")
SUBTITLES_DIR = os.path.join(BASE_DIR, "subtitles")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")

def clean_folder(folder, keep=False):
    if not os.path.exists(folder):
        return
    if keep:
        print(f"Keeping files in {folder}")
        return
    for filename in os.listdir(folder):
        file_path = os.path.join(folder, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            print(f"Failed to delete {file_path}: {e}")

print("Cleaning up intermediate files...")
clean_folder(VIDEOS_DIR, keep=False)
clean_folder(AUDIOS_DIR, keep=False)
clean_folder(SUBTITLES_DIR, keep=False)   
# clean_folder(OUTPUTS_DIR, keep=True)    # uncomment to also delete outputs

print("Cleanup completed. Final videos are in:", OUTPUTS_DIR)